# Count Objects in Real Time with a Webcam — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Count Objects in Real Time with a Webcam** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/webcam-object-counter/detect_image.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/webcam-object-counter/detect_image.py) —
object detection and counting with OpenCV and a pretrained YOLO11n model.

**Important: this notebook does NOT access a live webcam.** Hosted notebook environments like Colab, Kaggle, and
Binder run in the cloud, with no route to a camera physically attached to your computer — there is no way for
browser-based Python code here to open your webcam the way `cv2.VideoCapture(0)` does on your own machine. This
notebook instead runs the exact same detection code on two small **bundled sample images**, so you can see the
model work end-to-end without any hardware. For the real, live-webcam experience (Step 4 of the lesson), you need
the local `uv` path — see the project doc's "Where to run this" section.

Run the cells in order.

In [ ]:
!pip install -q opencv-python-headless ultralytics

## Download the bundled sample images

The same two images shipped in `examples/webcam-object-counter/samples/` in the course repo.

In [ ]:
import urllib.request

BASE_URL = (
    "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/"
    "main/examples/webcam-object-counter/samples/"
)
for filename in ["street.jpg", "people.jpg"]:
    urllib.request.urlretrieve(BASE_URL + filename, filename)
print("Downloaded sample images.")

## Load the pretrained model

`yolo11n.pt` ("n" = nano) is the smallest YOLO11 checkpoint — a few megabytes, pretrained on the COCO dataset (80
everyday object classes: person, car, dog, bus, ...). `ultralytics` downloads it automatically the first time you
construct `YOLO(...)`, no signup or API key involved.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

## Run detection on the sample images and count a target class

In [ ]:
TARGET_CLASS = "person"

running_total = 0
for image_path in ["street.jpg", "people.jpg"]:
    result = model(image_path, verbose=False)[0]
    count = sum(1 for box in result.boxes if model.names[int(box.cls)] == TARGET_CLASS)
    running_total += count
    print(f"{image_path}: {count} {TARGET_CLASS}(s) -- running total: {running_total}")

print(f"\nTotal {TARGET_CLASS}(s) across both images: {running_total}")

## Show an annotated image with bounding boxes

In [ ]:
from PIL import Image

result = model("street.jpg")[0]
annotated = result.plot()  # BGR numpy array, OpenCV's default channel order
Image.fromarray(annotated[:, :, ::-1])  # flip to RGB for display in the notebook

That's the same detection-and-counting logic Steps 1 and 2 of the lesson walk through, plus the frame-by-frame
version in Step 3 for video. Step 4 — a real, live webcam feed — only works locally; see the project doc for why,
and for `detect_webcam.py`, the script that does it.